In [1]:
from mofapy2.run.entry_point import entry_point
import pandas as pd
import numpy as np
import mofax as mfx
import os

# initialise the entry point
ent = entry_point()


        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | |  | | |__| | | / ____ \|_|            ###
        ###          |_|  |_|\____/|_|/_/    \_\              ###
        ###                                                   ### 
        ######################################################### 
       
 
        


In [2]:
INPUT_FOLDER = "PATH_TO_INPUT_FOLDER"

In [9]:
# --- Read data ---
proteomics = pd.read_csv(os.path.join(INPUT_FOLDER, "Proteomics_view.csv"))
transcriptomics = pd.read_csv(os.path.join(INPUT_FOLDER, "RNA_view.csv"))

# Move "Gene" column into index (like rownames in R)
proteomics_mat = proteomics.set_index("Gene")
transcriptomics_mat = transcriptomics.set_index("Gene")

# --- Reshape to long format ---
proteomics_long = (
    proteomics_mat
    .reset_index()
    .melt(id_vars="Gene", var_name="sample", value_name="value")
    .rename(columns={"Gene": "feature"})
)
proteomics_long["view"] = "Proteomics"
# Make features unique by adding suffix
proteomics_long["feature"] = proteomics_long["feature"].astype(str) + "_prot"

transcriptomics_long = (
    transcriptomics_mat
    .reset_index()
    .melt(id_vars="Gene", var_name="sample", value_name="value")
    .rename(columns={"Gene": "feature"})
)
transcriptomics_long["view"] = "Transcriptomics"
# Make features unique by adding suffix
transcriptomics_long["feature"] = transcriptomics_long["feature"].astype(str) + "_rna"

# --- Combine into one long DataFrame ---
mofa_long = pd.concat([proteomics_long, transcriptomics_long], ignore_index=True)

mofa_long

,feature,sample,value,view
0,Eif2b4_prot,05J_C10_A10,-1.636681,Proteomics
1,Slc38a2_prot,05J_C10_A10,-0.899462,Proteomics
2,Wnk1_prot,05J_C10_A10,2.877517,Proteomics
3,Krt6a_prot,05J_C10_A10,-1.962201,Proteomics
4,Rpl18_prot,05J_C10_A10,1.478620,Proteomics
...,...,...,...,...
198985,Cstf2_rna,07J_SVEC_D9,-0.088757,Transcriptomics
198986,Acsl1_rna,07J_SVEC_D9,-1.201857,Transcriptomics
198987,Tlcd3a_rna,07J_SVEC_D9,-0.855548,Transcriptomics
198988,Elp1_rna,07J_SVEC_D9,-0.036836,Transcriptomics


In [11]:
# Data.frame format
ent.set_data_df(mofa_long, likelihoods = ["gaussian","gaussian"])


No "group" column found in the data frame, we will assume a common group for all samples...


Loaded group='single_group' view='Proteomics' with N=67 samples and D=990 features...
Loaded group='single_group' view='Transcriptomics' with N=67 samples and D=1980 features...




In [13]:
ent.set_model_options(factors = 7)

Model options:
- Automatic Relevance Determination prior on the factors: False
- Automatic Relevance Determination prior on the weights: True
- Spike-and-slab prior on the factors: False
- Spike-and-slab prior on the weights: True
Likelihoods:
- View 0 (Proteomics): gaussian
- View 1 (Transcriptomics): gaussian




In [15]:
ent.set_train_options(
    convergence_mode = "slow", 
    dropR2 = 0, 
    gpu_mode = True, 
    seed = 1
)


GPU mode is activated



In [ ]:
ent.build()

ent.run()

In [ ]:
# Save the output
ent.save(outfile="PATH/TO/SCOTplus_trained_model.hdf5")